# Credit Scoring - Loan Approval Prediction

CodeAlpha ML Internship - Task 1

Goal: predict if a loan application gets approved (`Y`) or rejected (`N`) using the applicant's financial details.

## 1. Imports

In [ ]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report, confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

## 2. Load the dataset

In [ ]:
data = pd.read_csv("Loan Eligibility Prediction.csv")
data.head()

## 3. First look at the data

In [ ]:
print("Rows and columns:", data.shape)
print("\nColumn types:")
print(data.dtypes)
print("\nMissing values:")
print(data.isna().sum())
print("\nDuplicate rows:", data.duplicated().sum())

## 4. Quick EDA

Before training I wanted to see which columns actually matter. Credit history looks like the biggest one.

In [ ]:
print("Approved vs rejected:")
print(data["Loan_Status"].value_counts())

# approval rate for each group
for col in ["Credit_History", "Education", "Married", "Property_Area"]:
    print(f"\nApproval rate by {col}:")
    print((data.groupby(col)["Loan_Status"].apply(lambda s: (s == "Y").mean()) * 100).round(1))

## 5. Features and target

`Loan_Status` is what we predict. `Customer_ID` is just an ID number, it says nothing about the applicant, so I drop it.

In [ ]:
data = data.drop_duplicates().copy()
data = data.drop(columns=["Customer_ID"])
data = data.dropna(subset=["Loan_Status"])

X = data.drop(columns="Loan_Status")
y = data["Loan_Status"]

print("Input columns:", list(X.columns))

## 6. Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 7. Preprocessing

Numbers: fill missing with median, then scale.
Text columns: fill missing with most common value, then one-hot encode.

In [ ]:
numeric_columns = X.select_dtypes(include="number").columns
categorical_columns = X.select_dtypes(exclude="number").columns

numeric_steps = Pipeline(steps=[
    ("fill_missing", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_steps = Pipeline(steps=[
    ("fill_missing", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

prepare_columns = ColumnTransformer(transformers=[
    ("numbers", numeric_steps, numeric_columns),
    ("categories", categorical_steps, categorical_columns),
])

## 8. Train 3 models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

trained_models = {}
for model_name, model in models.items():
    model_pipeline = Pipeline(steps=[
        ("prepare", prepare_columns),
        ("classifier", model),
    ])
    model_pipeline.fit(X_train, y_train)
    trained_models[model_name] = model_pipeline
    print(model_name, "trained")

## 9. Compare results

`Y` (approved) is treated as the positive class. ROC-AUC uses the predicted probability of `Y`.

In [ ]:
results = []
for model_name, model_pipeline in trained_models.items():
    predictions = model_pipeline.predict(X_test)
    y_index = list(model_pipeline.classes_).index("Y")
    prob_yes = model_pipeline.predict_proba(X_test)[:, y_index]

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, pos_label="Y"),
        "Recall": recall_score(y_test, predictions, pos_label="Y"),
        "F1": f1_score(y_test, predictions, pos_label="Y"),
        "ROC-AUC": roc_auc_score(y_test == "Y", prob_yes),
    })

results_table = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False).round(4)
results_table

## 10. Best model in detail

In [ ]:
best_name = results_table.iloc[0]["Model"]
best_model = trained_models[best_name]
best_predictions = best_model.predict(X_test)

print("Best model (by ROC-AUC):", best_name)
print("\nConfusion matrix (rows = actual N, Y):")
print(confusion_matrix(y_test, best_predictions, labels=["N", "Y"]))
print("\nClassification report:")
print(classification_report(y_test, best_predictions, zero_division=0))

## 11. Try one prediction

In [ ]:
example_application = X_test.iloc[[0]]
print("Applicant details:")
display(example_application)
print("Predicted loan status:", best_model.predict(example_application)[0])
print("Actual loan status:", y_test.iloc[0])